# Phase 6: Formal Model Comparison & Business Trade-off Analysis

**Project:** Customer Churn Prediction & Retention Analytics  
**Models Compared:**
1. **Balanced Logistic Regression** (`models/logistic_regression_pipeline.joblib`)
2. **Balanced Random Forest Classifier** (`models/random_forest_pipeline.joblib`)

**Evaluation Set:** Identical 20% stratified test set ($N = 1,409$, 374 actual churners)  

### Objectives:
1. Load both saved model pipelines and evaluate against the exact same test dataset.
2. Generate a unified comparison table covering Accuracy, Precision, Recall, F1-score, ROC-AUC, and Confusion Matrix counts.
3. Export the comparison table to `outputs/model_comparison.csv`.
4. Plot grouped metric comparisons and joint ROC curves.
5. Quantify the exact trade-offs (detection rate vs. false positives) to select the optimal model for business retention.

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# Set visualization styles
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#cccccc"

sys.path.append("..")
from src.preprocess import get_train_test_data

fig_dir = Path("../outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Saved Pipelines & Test Dataset

In [ ]:
# Load test data
data = get_train_test_data(file_path="../data/processed/telco_churn_clean.csv")
X_test, y_test = data["X_test"], data["y_test"]

# Load trained pipelines
lr_pipeline = joblib.load("../models/logistic_regression_pipeline.joblib")
rf_pipeline = joblib.load("../models/random_forest_pipeline.joblib")

print(f"Test Set Instances: {len(y_test):,} (Actual Churners: {(y_test == 1).sum():,})")
print("Both pipelines loaded successfully.")

## 3. Compute Comparative Performance Metrics

In [ ]:
models = {
    "Logistic Regression": lr_pipeline,
    "Random Forest": rf_pipeline
}

rows = []
roc_dict = {}

for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_dict[name] = {"fpr": fpr, "tpr": tpr, "auc": auc}
    
    rows.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "ROC_AUC": round(auc, 4),
        "True_Negatives": int(tn),
        "False_Positives": int(fp),
        "False_Negatives": int(fn),
        "True_Positives": int(tp),
        "Total_Actual_Churners": int(tp + fn),
        "Detection_Rate_Pct": round(rec * 100, 2),
        "Miss_Rate_Pct": round((fn / (tp + fn)) * 100, 2)
    })

comp_df = pd.DataFrame(rows)
comp_df.to_csv(output_dir / "model_comparison.csv", index=False)
display(comp_df)

## 4. Grouped Metrics Comparison Chart

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"]
plot_df = pd.melt(comp_df, id_vars=["Model"], value_vars=metric_cols, var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model", palette=["#2980b9", "#27ae60"], ax=ax)
ax.set_title("Model Comparison: Key Performance Metrics (Test Set)", fontsize=14, pad=12, weight="bold")
ax.set_ylabel("Score (0.0 - 1.0)", fontsize=12)
ax.set_xlabel("Evaluation Metric", fontsize=12)
ax.set_ylim(0, 1.15)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.3f}",
            xy=(p.get_x() + p.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
            weight="bold"
        )

ax.legend(title="Model", loc="upper right")
plt.tight_layout()
plt.savefig(fig_dir / "model_comparison_metrics.png", dpi=300)
plt.show()

## 5. Joint ROC Curves Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {"Logistic Regression": "#2980b9", "Random Forest": "#27ae60"}

for name, res in roc_dict.items():
    ax.plot(res["fpr"], res["tpr"], color=colors[name], lw=2.5, label=f"{name} (AUC = {res['auc']:.4f})")

ax.plot([0, 1], [0, 1], color="#7f8c8d", lw=1.5, linestyle="--", label="Random Baseline (AUC = 0.5000)")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Recall)", fontsize=12)
ax.set_title("Model Comparison: ROC Curves (Test Set)", fontsize=14, pad=12, weight="bold")
ax.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.savefig(fig_dir / "model_roc_comparison.png", dpi=300)
plt.show()

## 6. Trade-off Analysis & Business Decision

### Quantified Differences:
- **Churners Correctly Detected:** Logistic Regression detected **292** vs. Random Forest **241** (**+51 churners detected** by Logistic Regression, a **+13.63 percentage point** recall advantage).
- **Churners Missed (False Negatives):** Logistic Regression missed **82** vs. Random Forest **133** (**51 fewer missed churners** with Logistic Regression).
- **False Positives (False Alarms):** Logistic Regression produced **280** vs. Random Forest **186** (**94 fewer false positives** with Random Forest).

### Business Recommendation:
- **Selected Preferred Model:** **Logistic Regression**
- **Justification:** In subscription churn analytics, the primary objective is proactive retention intervention. Failing to identify a churning customer (**False Negative**) results in the permanent loss of the customer's recurring revenue and lifetime value ($CLTV$). In contrast, reaching out to a customer who wasn't going to churn (**False Positive**) carries minimal intervention cost (e.g., automated email, survey, or tailored discount). 
- Because Logistic Regression delivers **higher Recall (78.07% vs. 64.44%)**, **higher F1-score (0.6173 vs. 0.6017)**, and **higher ROC-AUC (0.8489 vs. 0.8333)**, it is the superior choice for proactive customer retention.